# 7.3 U-Net、Band-Split 与 RoFormer 结构可视化


## 兼容性提示

- 请在前面章节已经建立的虚拟环境中运行本章 notebook（推荐 Python 3.11）。部分依赖包在 Python 3.13 上可能出现 `collections.Hashable` 等兼容性问题。
- 首次运行预训练模型时，工具会自动下载 checkpoint，需要一定时间，请耐心等待；本章不给出具体下载大小或耗时预估，因为不同网络与硬件差异较大。
- 若某模型依赖缺失，可将 `ENABLE_OPTIONAL_MODELS` 或对应运行开关设为 `0`，跳过该模型继续学习其余内容。


## 1. 环境准备


In [ ]:
import os
import sys
import tempfile
from pathlib import Path

# matplotlib/numba 缓存目录：用跨平台的系统临时目录
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "mplconfig"))
os.environ.setdefault("NUMBA_CACHE_DIR", str(Path(tempfile.gettempdir()) / "numba_cache"))

# 路径推断：从 cwd 向上找含 CODE/chapter07/_common 的目录；NOTEBOOK_DIR 指向 CODE/chapter07/
_p = Path.cwd()
while not (_p / "CODE" / "chapter07" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter07/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
NOTEBOOK_DIR = _p / "CODE" / "chapter07"
CODE_ROOT = NOTEBOOK_DIR.parent
REPO_ROOT = CODE_ROOT.parent
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

FIG_DIR = NOTEBOOK_DIR / "output_figures"
FIG_DIR.mkdir(exist_ok=True)

print("NOTEBOOK_DIR:", NOTEBOOK_DIR.relative_to(REPO_ROOT))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from chapter07.visualizations.architecture_diagrams import (
    SkipConnectionSpec,
    make_frequency_bands,
    plot_band_split_layout,
    plot_roformer_attention_sketch,
    plot_unet_shape_flow,
    rotary_embedding_angles,
    unet_shape_flow,
)


## 2. U-Net：用下采样扩大上下文，再用 skip connection 恢复细节


图中的 `B C F T` 是张量维度缩写：`B` 为 batch size，`C` 为 channel 数，`F` 为频率 bin 数，`T` 为时间帧数。

下面的 `UNET_PLOT_PARAMS` 暴露本图的主要绘图参数。若文字偏小，调 `font_scale` 或各类 `*_font_size`；若跳跃连线与文字或方块重叠，调 `SkipConnectionSpec` 的 `y_offset`、`curvature` 或 `label_y_offset`。


In [ ]:
stages = unet_shape_flow(
    batch=1,
    channels=2,
    freq_bins=256,
    frames=128,
    base_channels=16,
)

UNET_PLOT_PARAMS = {
    "font_scale": 2,
    "figsize": (13, 4.4),
    "stage_x_min": 0.06,
    "stage_x_max": 0.94,
    "stage_y": 0.50,
    "stage_width": 0.10,
    "min_stage_height": 0.18,
    "max_stage_height": 0.36,
    "stage_title_font_size": 9,
    "stage_shape_font_size": 8,
    "stage_role_font_size": 9,
    "skip_label_font_size": 8,
    "note_font_size": 8,
    "stage_title_y_offset": 0.01,
    "stage_role_y_offset": -0.02,
    "note_xy": (0.02, 0.10),
    "encoder_color": "0.30",
    "bottleneck_color": "0.50",
    "decoder_color": "0.70",
    "block_edge_color": "0.20",
    "arrow_color": "0.25",
    "arrow_linewidth": 1.1,
    "arrow_mutation_scale": 10,
    "skip_color": "0.35",
    "skip_linewidth": 1.0,
    "skip_connections": (
        SkipConnectionSpec(1, 5, y_offset=0.26, label="浅层细节", curvature=-0.14, label_y_offset=0.05),
        SkipConnectionSpec(2, 4, y_offset=-0.20, label="中层结构", curvature=0.18, label_y_offset=-0.15),
    ),
}

plot_unet_shape_flow(
    stages,
    FIG_DIR / "07_3_unet_shape_flow.png",
    **UNET_PLOT_PARAMS,
)
plt.show()

for stage in stages:
    print(f"{stage.name:12s} {stage.shape}  {stage.role}")


在频谱分离中，U-Net 通常输出 mask，而不是直接输出 waveform。下采样让模型看到更长时间和更宽频率上下文；skip connection 把高分辨率局部细节送回 decoder。


## 3. Tiny U-Net forward：只检查 shape，不训练


In [ ]:
try:
    import torch
    from chapter07.visualizations.toy_models import TinySpectrogramUNet
except ImportError as exc:
    print("PyTorch is not available; skipping toy forward.")
    print(exc)
else:
    torch.manual_seed(0)
    model = TinySpectrogramUNet(in_channels=2, base_channels=8)
    x = torch.randn(1, 2, 256, 128)
    with torch.no_grad():
        y, shapes = model(x, return_shapes=True)
    print("input == output shape:", tuple(x.shape) == tuple(y.shape))
    for name, shape in shapes.items():
        print(f"{name:10s} {shape}")


## 4. Band-split：先分频带，再做局部建模与跨带融合


左侧子图按真实 `frequency bin` 高度展示非均匀频带，因此低频条带会很窄；频带名称和范围放在 legend 中，避免文字挤在窄条带里。右侧结构图使用独立的 `BAND_SPLIT_PLOT_PARAMS`，不会继承 U-Net 的字号。


In [ ]:
bands = make_frequency_bands(n_bins=1024)
BAND_SPLIT_PLOT_PARAMS = {
    "font_scale": 1.6,
    "figsize": (13, 4.8),
    "width_ratios": (1.05, 1.25),
    "title_font_size": 11,
    "axis_label_font_size": 10,
    "tick_label_font_size": 8,
    "legend_font_size": 8,
    "legend_title_font_size": 9,
    "block_font_size": 8,
    "left_title": "非均匀频带",
    "right_title": "逐频带建模与跨带融合",
    "legend_title": "频带范围",
    "legend_loc": "upper left",
    "legend_bbox_to_anchor": (1.02, 1.0),
    "band_alpha": 0.82,
    "band_edge_color": "0.96",
    "block_height": 0.09,
    "band_block_xy": (0.08, 0.17),
    "local_block_xy": (0.32, 0.17),
    "merge_block_xywh": (0.80, 0.41, 0.18, 0.13),
    "right_y_top": 0.82,
    "right_y_bottom": 0.18,
    "arrow_color": "0.25",
    "arrow_linewidth": 1.1,
    "arrow_mutation_scale": 10,
    "band_to_local_arrow_gap": 0.005,
    "local_to_merge_arrow_start_gap": 0.02,
    "local_to_merge_arrow_end_gap": 0.01,
}

plot_band_split_layout(
    bands,
    n_frames=96,
    out_path=FIG_DIR / "07_3_band_split_layout.png",
    **BAND_SPLIT_PLOT_PARAMS,
)
plt.show()

for band in bands:
    print(f"{band.name:8s} bins {band.start_bin:4d} - {band.end_bin:4d}")


Band-split 方法不把整个频谱当成一个均匀网格处理，而是让低频有更细的建模入口，高频用更宽的 band 聚合。这样能减少计算量，也更贴近音乐频率分布。


## 5. RoFormer：用旋转位置编码给注意力提供相对位置信息


In [ ]:
angles = rotary_embedding_angles(seq_len=64, dim=16)
ROFORMER_PLOT_PARAMS = {
    "font_scale": 1.6,
    "figsize": (12, 3.8),
    "title_font_size": 11,
    "axis_label_font_size": 10,
    "legend_font_size": 7,
    "node_font_size": 10,
    "center_text_font_size": 8,
    "line_width": 1.0,
    "token_grid_shape": (8, 12),
    "token_grid_active_rows": (2, 6),
    "token_grid_active_cols": (3, 9),
    "token_inactive_face_color": "0.94",
    "token_active_face_color": "0.62",
    "token_edge_color": "0.35",
    "token_linewidth": 0.7,
    "token_tick_label_font_size": 8,
    "rope_legend_loc": "upper left",
    "rope_legend_bbox_to_anchor": (1.02, 1.0),
    "q_node_xy": (0.18, 0.50),
    "k_upper_xy": (0.50, 0.72),
    "k_lower_xy": (0.50, 0.28),
    "v_node_xy": (0.82, 0.50),
    "center_text_xy": (0.50, 0.50),
    "arrow_color": "0.25",
    "arrow_linewidth": 1.1,
    "arrow_mutation_scale": 10,
}

plot_roformer_attention_sketch(
    seq_len=64,
    dim=16,
    out_path=FIG_DIR / "07_3_roformer_attention_sketch.png",
    **ROFORMER_PLOT_PARAMS,
)
plt.show()

print("RoPE angle matrix:", angles.shape)
print("first token angles:", np.round(angles[0, :4], 4))
print("second token angles:", np.round(angles[1, :4], 4))


RoFormer 的重点是把 Transformer 的注意力和相对位置信息结合起来。用于音源分离时，它通常还会和 band-split 或 mel-band 表示结合，避免在完整频谱上做过重的全局注意力。


## 6. 小结

- U-Net 解释 mask learning 的多尺度路径。
- Band-split 解释为什么现代模型常按频带组织计算。
- RoFormer / RoPE 解释注意力如何获得位置关系。
